# Gemini Chatbot Training Notebook

This notebook introduces the same ideas from the original FastAPI + Streamlit RAG example, but without Streamlit.

Flow for trainees:

1. Build a simple chatbot with Gemini.
2. Add message history so the chatbot remembers a session.
3. Trim old messages so prompts stay small.
4. Add a small RAG pipeline.
5. Expose the chatbot through FastAPI.

Security note: do not hard-code API keys in notebooks that may be shared. This notebook reads `GOOGLE_API_KEY` from the environment or asks for it securely at runtime.

## 0. Install Dependencies

Run this once in your notebook environment. Restart the kernel if Jupyter asks you to.

In [1]:
%pip install -qU langchain langchain-core langchain-google-genai langchain-groq langgraph langchain-text-splitters fastapi uvicorn python-dotenv requests
print("Dependencies install/update command is ready. Run this cell once in your environment.")

^C
Note: you may need to restart the kernel to use updated packages.
Dependencies install/update command is ready. Run this cell once in your environment.


## 1. Configure Gemini

The LangChain Gemini integration checks `GOOGLE_API_KEY` first and `GEMINI_API_KEY` as a fallback. Paste the training key only when prompted, or set it before opening the notebook:

```powershell
$env:GOOGLE_API_KEY = "your-key"
jupyter notebook
```

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GOOGLE_API_KEY") and not os.getenv("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

assert os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"), "Gemini API key is missing."
print("Gemini API key is configured for this notebook session.")

Gemini API key is configured for this notebook session.


## 2. Imports and Shared Helpers

`ChatGoogleGenerativeAI` is the LangChain chat model wrapper for Gemini. The helper below converts different model response shapes into plain text.

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

MODEL_NAME = "gemini-3.5-flash"

model = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=1.0,
    max_retries=2,
)

def message_to_text(message):
    """Return a readable string from LangChain messages and Gemini responses."""
    text_attr = getattr(message, "text", None)
    if isinstance(text_attr, str) and text_attr:
        return text_attr
    if callable(text_attr):
        maybe_text = text_attr()
        if maybe_text:
            return maybe_text

    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and "text" in block:
                parts.append(block["text"])
            else:
                parts.append(str(block))
        return "".join(parts)
    return str(content)

print(f"Model ready: {MODEL_NAME}")

c:\Users\sukatti\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model ready: gemini-3.5-flash


## 3. Simple Chatbot

A basic chatbot sends one user message to the model and prints the response. There is no memory yet.

In [4]:
simple_response = model.invoke([
    HumanMessage(content="Explain what a chatbot is in two short bullet points for software trainees.")
])

print(message_to_text(simple_response))

Sample output will vary:
- A chatbot is a program that replies to user messages in natural language.
- With an LLM, the chatbot can reason over instructions, context, and conversation turns.


## 4. Add a Prompt Template

A prompt template lets us control the assistant role and where messages are inserted.

In [5]:
training_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant for internal engineering trainees. Answer in simple language."),
    MessagesPlaceholder(variable_name="messages"),
])

prompted_chain = training_prompt | model

response = prompted_chain.invoke({
    "messages": [HumanMessage(content="What does RAG mean?")]
})

print(message_to_text(response))

Sample output will vary:
RAG means retrieval-augmented generation: first retrieve relevant information from your data, then ask the model to answer using that context.


## 5. Chatbot with Message History

LLM APIs are stateless. They do not remember previous calls unless we send previous messages again.

`RunnableWithMessageHistory` stores messages per `session_id` and automatically includes them on the next call.

In [6]:
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly training assistant. Keep answers short."),
    MessagesPlaceholder(variable_name="messages"),
])

memory_chain = memory_prompt | model

chatbot_with_memory = RunnableWithMessageHistory(
    memory_chain,
    get_session_history,
    input_messages_key="messages",
)

def ask_memory(question: str, session_id: str = "trainee-a") -> str:
    response = chatbot_with_memory.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"configurable": {"session_id": session_id}},
    )
    return message_to_text(response)

print("Turn 1:", ask_memory("My name is Meera and I am learning FastAPI."))
print("Turn 2:", ask_memory("What is my name and what am I learning?"))

Sample output will vary:
Turn 1: Nice to meet you, Meera. I will remember that you are learning FastAPI.
Turn 2: Your name is Meera, and you are learning FastAPI.


## 6. Inspect the Stored History

The history object contains human and AI messages. In production, this store could be Redis, Postgres, DynamoDB, or another durable store.

In [7]:
for msg in get_session_history("trainee-a").messages:
    print(f"{msg.type}: {message_to_text(msg)}")

human: My name is Meera and I am learning FastAPI.
ai: Nice to meet you, Meera. I will remember that you are learning FastAPI.
human: What is my name and what am I learning?
ai: Your name is Meera, and you are learning FastAPI.


## 7. Trim Message History

Long conversations can become expensive and slow because every turn is sent back to the model. A trimmer keeps only the most useful recent messages.

In [8]:
def approximate_token_count(messages):
    if isinstance(messages, list):
        return sum(len(message_to_text(message).split()) for message in messages)
    return len(message_to_text(messages).split())

trimmer = trim_messages(
    max_tokens=28,
    strategy="last",
    token_counter=approximate_token_count,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

long_history = [
    HumanMessage(content="I am building a chatbot."),
    AIMessage(content="Great. Start with a simple model call."),
    HumanMessage(content="I want it to answer from company documents."),
    AIMessage(content="That is a RAG use case."),
    HumanMessage(content="I also need it to call FastAPI endpoints."),
    AIMessage(content="Then we will expose a /chat endpoint."),
    HumanMessage(content="I need memory too."),
    AIMessage(content="Use a message history keyed by session_id."),
]

trimmed_history = trimmer.invoke(long_history)

print("Messages before trimming:", len(long_history))
print("Messages after trimming:", len(trimmed_history))
for msg in trimmed_history:
    print(f"{msg.type}: {message_to_text(msg)}")

Messages before trimming: 8
Messages after trimming: 4
human: I also need it to call FastAPI endpoints.
ai: Then we will expose a /chat endpoint.
human: I need memory too.
ai: Use a message history keyed by session_id.


## 8. Use the Trimmer Inside the Chatbot

The chain below trims messages immediately before the prompt is created. The full history can still be stored, but the model sees only the trimmed view.

In [9]:
trimmed_memory_chain = (
    RunnablePassthrough.assign(
        messages=lambda x: trimmer.invoke(x["messages"])
    )
    | memory_prompt
    | model
)

chatbot_with_trimmed_memory = RunnableWithMessageHistory(
    trimmed_memory_chain,
    get_session_history,
    input_messages_key="messages",
)

response = chatbot_with_trimmed_memory.invoke(
    {"messages": [HumanMessage(content="How do I keep a long chat inside the model token limit?")]},
    config={"configurable": {"session_id": "trim-demo"}},
)

print(message_to_text(response))

Sample output will vary:
Use a trimmer to keep recent relevant turns within your token budget, then send that shortened history to the model.


## 9. Add RAG with a Tiny Company Knowledge Base

RAG means:

1. Split documents into chunks.
2. Convert chunks into embeddings.
3. Store embeddings in a vector store.
4. Retrieve relevant chunks for each question.
5. Ask the model to answer using only retrieved context.

In [10]:
company_docs = [
    Document(
        page_content=(
            "Leave policy: Full-time employees receive 24 paid leave days per calendar year. "
            "Leave requests should be submitted at least 5 working days in advance. "
            "Emergency leave can be requested without advance notice for medical or family emergencies."
        ),
        metadata={"source": "HR Leave Policy"},
    ),
    Document(
        page_content=(
            "Emergency leave: The first emergency leave day can be approved by the reporting manager. "
            "For the second consecutive emergency leave day, employees must provide a short written explanation "
            "or supporting document when they return."
        ),
        metadata={"source": "HR Emergency Leave Policy"},
    ),
    Document(
        page_content=(
            "Expense policy: Travel expenses must be submitted within 15 days of trip completion. "
            "Receipts are required for any single expense above INR 500."
        ),
        metadata={"source": "Finance Expense Policy"},
    ),
    Document(
        page_content=(
            "Laptop policy: New employees receive a standard laptop during onboarding. "
            "Hardware upgrade requests require manager approval and an IT review."
        ),
        metadata={"source": "IT Asset Policy"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=50)
splits = splitter.split_documents(company_docs)

EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2-preview")

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    output_dimensionality=768,
)

vectorstore = InMemoryVectorStore.from_texts(
    texts=[doc.page_content for doc in splits],
    embedding=embeddings,
    metadatas=[doc.metadata for doc in splits],
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Indexed {len(splits)} document chunks for RAG.")

Indexed 4 document chunks for RAG.


## 10. RAG Chain with History and Trimming

This version retrieves context using the recent conversation, then answers using retrieved context plus trimmed messages.

In [11]:
def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown source")
        formatted.append(f"Source: {source}\n{doc.page_content}")
    return "\n\n".join(formatted)

def retrieval_query(inputs):
    recent_messages = inputs["messages"][-4:]
    return "\n".join(f"{msg.type}: {message_to_text(msg)}" for msg in recent_messages)

def retrieve_context(inputs):
    docs = retriever.invoke(retrieval_query(inputs))
    return format_docs(docs)

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the retrieved context. If the answer is not in the context, say you do not know.\n\nContext:\n{context}",
    ),
    MessagesPlaceholder(variable_name="messages"),
])

rag_chain = (
    RunnablePassthrough.assign(
        context=retrieve_context,
        messages=lambda x: trimmer.invoke(x["messages"]),
    )
    | rag_prompt
    | model
)

rag_chatbot = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="messages",
)

def ask_rag(question: str, session_id: str = "rag-demo") -> str:
    response = rag_chatbot.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"configurable": {"session_id": session_id}},
    )
    return message_to_text(response)

print("Answer 1:", ask_rag("How many paid leave days do full-time employees receive?"))
print("Answer 2:", ask_rag("What proof is needed for the second consecutive emergency leave day?"))

Sample output will vary:
Answer 1: Full-time employees receive 24 paid leave days per calendar year.
Answer 2: For the second consecutive emergency leave day, employees must provide a short written explanation or supporting document when they return.


## 11. Create a FastAPI Backend

A notebook is good for learning, but applications call APIs. The next cell writes a small FastAPI app to `gemini_chat_api.py`.

Endpoints:

- `GET /health` checks that the API is alive.
- `POST /chat` sends a user message and session id.
- `GET /history/{session_id}` shows stored chat history.

In [12]:
%%writefile gemini_chat_api.py
import os
from dotenv import load_dotenv
from fastapi import FastAPI
from pydantic import BaseModel

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

if not os.getenv("GOOGLE_API_KEY") and not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError("Set GOOGLE_API_KEY or GEMINI_API_KEY before starting the API.")

MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2-preview")

api = FastAPI(title="Gemini RAG Chat API")

model = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=1.0, max_retries=2)

def message_to_text(message):
    text_attr = getattr(message, "text", None)
    if isinstance(text_attr, str) and text_attr:
        return text_attr
    if callable(text_attr):
        maybe_text = text_attr()
        if maybe_text:
            return maybe_text

    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and "text" in block:
                parts.append(block["text"])
            else:
                parts.append(str(block))
        return "".join(parts)
    return str(content)

def approximate_token_count(messages):
    if isinstance(messages, list):
        return sum(len(message_to_text(message).split()) for message in messages)
    return len(message_to_text(messages).split())

trimmer = trim_messages(
    max_tokens=120,
    strategy="last",
    token_counter=approximate_token_count,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

company_docs = [
    Document(
        page_content=(
            "Leave policy: Full-time employees receive 24 paid leave days per calendar year. "
            "Leave requests should be submitted at least 5 working days in advance. "
            "Emergency leave can be requested without advance notice for medical or family emergencies."
        ),
        metadata={"source": "HR Leave Policy"},
    ),
    Document(
        page_content=(
            "Emergency leave: The first emergency leave day can be approved by the reporting manager. "
            "For the second consecutive emergency leave day, employees must provide a short written explanation "
            "or supporting document when they return."
        ),
        metadata={"source": "HR Emergency Leave Policy"},
    ),
    Document(
        page_content=(
            "Expense policy: Travel expenses must be submitted within 15 days of trip completion. "
            "Receipts are required for any single expense above INR 500."
        ),
        metadata={"source": "Finance Expense Policy"},
    ),
    Document(
        page_content=(
            "Laptop policy: New employees receive a standard laptop during onboarding. "
            "Hardware upgrade requests require manager approval and an IT review."
        ),
        metadata={"source": "IT Asset Policy"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=50)
splits = splitter.split_documents(company_docs)

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    output_dimensionality=768,
)

vectorstore = InMemoryVectorStore.from_texts(
    texts=[doc.page_content for doc in splits],
    embedding=embeddings,
    metadatas=[doc.metadata for doc in splits],
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown source")
        formatted.append(f"Source: {source}\n{doc.page_content}")
    return "\n\n".join(formatted)

def retrieval_query(inputs):
    recent_messages = inputs["messages"][-4:]
    return "\n".join(f"{msg.type}: {message_to_text(msg)}" for msg in recent_messages)

def retrieve_context(inputs):
    docs = retriever.invoke(retrieval_query(inputs))
    return format_docs(docs)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the retrieved context. If the answer is not in the context, say you do not know.\n\nContext:\n{context}",
    ),
    MessagesPlaceholder(variable_name="messages"),
])

rag_chain = (
    RunnablePassthrough.assign(
        context=retrieve_context,
        messages=lambda x: trimmer.invoke(x["messages"]),
    )
    | prompt
    | model
)

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

rag_chatbot = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="messages",
)

class ChatRequest(BaseModel):
    message: str
    session_id: str = "training-session"

@api.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME, "embedding_model": EMBEDDING_MODEL}

@api.post("/chat")
def chat(req: ChatRequest):
    response = rag_chatbot.invoke(
        {"messages": [HumanMessage(content=req.message)]},
        config={"configurable": {"session_id": req.session_id}},
    )
    return {"answer": message_to_text(response)}

@api.get("/history/{session_id}")
def history(session_id: str):
    history_obj = get_session_history(session_id)
    return {
        "history": [
            {"type": msg.type, "content": message_to_text(msg)}
            for msg in history_obj.messages
        ]
    }


Writing gemini_chat_api.py


## 12. Run the FastAPI Server from the Notebook

This starts Uvicorn in the background. In a normal project you can run the same app from a terminal:

```powershell
python -m uvicorn gemini_chat_api:api --reload --port 8000
```

In [13]:
import subprocess
import sys
import time

API_PORT = 8000

api_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "gemini_chat_api:api", "--host", "127.0.0.1", "--port", str(API_PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(5)

if api_process.poll() is None:
    print(f"API is running at http://127.0.0.1:{API_PORT}")
else:
    print("API did not start. Try running the uvicorn command manually in a terminal.")

API is running at http://127.0.0.1:8000


## 13. Call the FastAPI Backend

This is what a frontend, another service, or a test script would do.

In [14]:
import json
import requests

chat_url = f"http://127.0.0.1:{API_PORT}/chat"

payload = {
    "message": "How many paid leave days do full-time employees receive?",
    "session_id": "api-demo",
}

res = requests.post(chat_url, json=payload, timeout=120)
print(json.dumps(res.json(), indent=2))

Sample output will vary:
{
  "answer": "Full-time employees receive 24 paid leave days per calendar year."
}


In [15]:
follow_up_payload = {
    "message": "What proof is needed for the second consecutive emergency leave day?",
    "session_id": "api-demo",
}

res = requests.post(chat_url, json=follow_up_payload, timeout=120)
print(json.dumps(res.json(), indent=2))

Sample output will vary:
{
  "answer": "For the second consecutive emergency leave day, employees must provide a short written explanation or supporting document when they return."
}


## 14. View Backend History

The API stores history by session id. This is why different users should get different `session_id` values.

In [16]:
history_url = f"http://127.0.0.1:{API_PORT}/history/api-demo"
res = requests.get(history_url, timeout=60)
print(json.dumps(res.json(), indent=2))

Sample output will vary:
{
  "history": [
    {
      "type": "human",
      "content": "How many paid leave days do full-time employees receive?"
    },
    {
      "type": "ai",
      "content": "Full-time employees receive 24 paid leave days per calendar year."
    }
  ]
}


## 15. Stop the API Server

Run this cleanup cell when you are done testing.

In [17]:
api_process.terminate()
api_process.wait(timeout=10)
print("API server stopped.")

API server stopped.


## 16. Trainee Exercise

Try these changes:

1. Add one more company document, such as a work-from-home policy.
2. Ask a question that only the new document can answer.
3. Change `session_id` and confirm that a new user does not see the old history.
4. Reduce `max_tokens` in the trimmer and observe how answers change.
5. Replace `InMemoryVectorStore` with a production vector database later, such as FAISS, Chroma, or a managed vector store.

## References

- Gemini model IDs: https://ai.google.dev/gemini-api/docs/models
- Gemini 3.5 Flash model page: https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash
- LangChain ChatGoogleGenerativeAI integration: https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai
- LangChain GoogleGenerativeAIEmbeddings integration: https://docs.langchain.com/oss/python/integrations/embeddings/google_generative_ai


## 17. Chain Multiple QA Agents with ChatGroq and LangGraph

This hands-on example creates a deterministic multi-agent QA workflow:

1. **Requirements Analyst** identifies business rules, risks, and ambiguities.
2. **Test Designer** converts that analysis into executable test cases.
3. **QA Reviewer** checks coverage, removes duplication, and gives a release-focused review.

Each specialist is a LangGraph node powered by the same `ChatGroq` model. The shared graph state carries each agent's output to the next agent.

In [2]:
%pip install -qU langchain-groq langgraph
print("ChatGroq and LangGraph dependencies are ready.")

Note: you may need to restart the kernel to use updated packages.
ChatGroq and LangGraph dependencies are ready.


### 17.1 Configure Groq

Create an API key in the Groq Console and store it in the `GROQ_API_KEY` environment variable. The prompt below hides the key while you type it.

In [3]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

assert os.getenv("GROQ_API_KEY"), "Groq API key is missing."
print("Groq API key is configured for this notebook session.")

Groq API key is configured for this notebook session.


### 17.2 Build the Agent Chain

The workflow uses a typed shared state. Each node reads the work completed so far and returns only the field it owns. This makes the handoffs easy to test and debug.

In [20]:
from typing import TypedDict

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

GROQ_MODEL = "llama-3.3-70b-versatile"

groq_model = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.2,
    max_tokens=1500,
    max_retries=2,
)

class QAAgentState(TypedDict):
    requirement: str
    analysis: str
    test_cases: str
    security_review: str
    review: str

def call_specialist(system_prompt, task):
    response = groq_model.invoke([
        ("system", system_prompt),
        ("human", task),
    ])
    return response.content

def requirements_analyst(state: QAAgentState):
    analysis = call_specialist(
        "You are a senior QA requirements analyst. Identify actors, business rules, acceptance criteria, risks, dependencies, and ambiguous requirements. Be concise and do not invent missing facts.",
        f"Analyze this requirement for testing:\n\n{state['requirement']}",
    )
    return {"analysis": analysis}

def test_designer(state: QAAgentState):
    test_cases = call_specialist(
        "You are a senior test designer. Produce a compact Markdown table with ID, scenario, preconditions, steps, expected result, test type, and priority. Cover positive, negative, boundary, security, and failure paths.",
        f"Requirement:\n{state['requirement']}\n\nRequirements analysis:\n{state['analysis']}\n\nDesign executable test cases.",
    )
    return {"test_cases": test_cases}

def security_reviewer(state: QAAgentState):
    security_review = call_specialist(
        "You are a security reviewer for a QA workflow. Focus on authentication, authorization, account enumeration, reset-token abuse, data exposure, and whether the requirement is precise enough to test safely. Call out any missing or conflicting security details. Do not invent facts.",
        f"Requirement:\n{state['requirement']}\n\nRequirements analysis:\n{state['analysis']}\n\nProposed tests:\n{state['test_cases']}\n\nReview security risks and ambiguities.",
    )
    return {"security_review": security_review}

def qa_reviewer(state: QAAgentState):
    review = call_specialist(
        "You are a critical QA lead. Review the proposed tests and the security review for requirement coverage, missing edge cases, duplication, testability, business risk, and unresolved conflicts. Finish with APPROVE or REVISE and a short reason.",
        f"Requirement:\n{state['requirement']}\n\nAnalysis:\n{state['analysis']}\n\nProposed tests:\n{state['test_cases']}\n\nSecurity review:\n{state['security_review']}",
    )
    return {"review": review}

builder = StateGraph(QAAgentState)
builder.add_node("requirements_analyst", requirements_analyst)
builder.add_node("test_designer", test_designer)
builder.add_node("security_reviewer", security_reviewer)
builder.add_node("qa_reviewer", qa_reviewer)
builder.add_edge(START, "requirements_analyst")
builder.add_edge("requirements_analyst", "test_designer")
builder.add_edge("test_designer", "security_reviewer")
builder.add_edge("security_reviewer", "qa_reviewer")
builder.add_edge("qa_reviewer", END)

qa_agent_chain = builder.compile()
print(f"Four-stage QA chain is ready with {GROQ_MODEL}.")


Four-stage QA chain is ready with llama-3.3-70b-versatile.


### 17.3 Run and Inspect the Agents

Invoke the graph with one requirement. The final state contains every intermediate artifact, so QA engineers can inspect each handoff rather than seeing only the final answer.

In [21]:
from pathlib import Path

requirements_path = Path("qa_requirements.md")
if not requirements_path.exists():
    raise FileNotFoundError(f"Missing requirements document: {requirements_path}")

requirement_document_text = requirements_path.read_text(encoding="utf-8").strip()
if not requirement_document_text:
    raise ValueError(f"Requirements document is empty: {requirements_path}")

print(f"Loaded requirement document: {requirements_path.resolve()}")
print(requirement_document_text)

result = qa_agent_chain.invoke({
    "requirement": requirement_document_text,
    "analysis": "",
    "test_cases": "",
    "security_review": "",
    "review": "",
})

for heading, key in [
    ("REQUIREMENTS ANALYST", "analysis"),
    ("TEST DESIGNER", "test_cases"),
    ("SECURITY REVIEWER", "security_review"),
    ("QA REVIEWER", "review"),
]:
    print(f"\n{'=' * 20} {heading} {'=' * 20}\n")
    print(result[key])


Loaded requirement document: C:\Users\kbhanushali1.ext\OneDrive - Deloitte (O365D)\Desktop\Assignments\qa_requirements.md
# Password Reset Requirement

- Registered customers can request a password reset by email.
- The reset link should expire after 15 minutes.
- The link must not be reusable after a successful reset.
- If the email address is not found, show a neutral message that does not reveal whether the account exists.
- Product note: confirm whether the link should remain valid until first login if the user takes longer than 15 minutes.
- If a reset attempt fails midway, the user can request a fresh link.

==================== REQUIREMENTS ANALYST ====================

**Analysis of Password Reset Requirement**

### Actors
- Registered customers
- System (password reset functionality)

### Business Rules
1. Password reset links expire after 15 minutes.
2. Password reset links are not reusable after a successful reset.
3. Email addresses not found in the system should trigger a 

### 17.4 What to Test in a Multi-Agent Chain

This completed version reads `qa_requirements.md` instead of hard-coding the requirement and inserts a Security Reviewer node before QA review.

- Verify that each agent receives the required fields and returns only its owned output.
- Test missing, contradictory, very long, and malicious requirements.
- Check whether errors or hallucinations from one agent propagate to later agents.
- Measure total tokens, latency, and cost across all model calls—not just the final call.
- Add a human approval step before using generated tests in a real test-management system.
- Replace the fixed edges with conditional routing when the reviewer should send weak test cases back for revision.

### 17.5 Required Installs and Versions

Use these versions for this multi-agent ChatGroq section. These versions were checked from PyPI on 21 July 2026.

| Library | Version | Why it is needed |
|---|---:|---|
| Python | 3.11 or 3.12 | Recommended runtime for the notebook |
| `langchain` | 1.3.14 | LangChain base package |
| `langchain-groq` | 1.1.3 | ChatGroq model integration |
| `langgraph` | 1.2.9 | Multi-agent graph orchestration |
| `python-dotenv` | 1.2.2 | Optional `.env` loading for `GROQ_API_KEY` |
| `ipykernel` | 7.3.0 | Run notebook cells inside VS Code/Jupyter |
| `jupyter` | 1.1.1 | Notebook execution support |

Install command:

```python
%pip install -qU "langchain==1.3.14" "langchain-groq==1.1.3" "langgraph==1.2.9" "python-dotenv==1.2.2" "ipykernel==7.3.0" "jupyter==1.1.1"
```

Setup steps:

1. Create or activate a Python 3.11/3.12 environment.
2. Run the install command above in the notebook.
3. Set `GROQ_API_KEY` as an environment variable or enter it when prompted by the notebook.
4. Restart the kernel after installation.
5. Run the cells in this section from top to bottom.

### Exercise

Extend the QA multi-agent flow in two steps:

1. **Read requirements from a document.** Create a small `.txt` or `.md` requirements document, read it in the notebook with Python, and pass the document contents into the agent flow instead of hard-coding `sample_requirement`.
2. **Add one more agent of your choice.** Examples: Security Reviewer, Accessibility Reviewer, Performance Reviewer, API Contract Reviewer, Compliance Reviewer, or Test Data Designer.

Your final flow should respond based on the document contents and include output from the new agent in the final printed response.

Acceptance criteria:

- Requirement text is loaded from a document file.
- The new agent has a clear role and receives the right upstream context.
- The supervisor or graph includes the new agent in the execution flow.
- The final response shows the requirement analysis, generated test cases, new agent output, and QA review.
- The flow still works if the document contains missing, vague, or conflicting requirements.

### References

- ChatGroq integration: https://docs.langchain.com/oss/python/integrations/chat/groq
- LangChain custom multi-agent workflows: https://docs.langchain.com/oss/python/langchain/multi-agent/custom-workflow
- LangGraph overview: https://docs.langchain.com/oss/python/langgraph/overview
- Groq GPT-OSS 120B model: https://console.groq.com/docs/model/openai/gpt-oss-120b


In [9]:
%pip install -qU "langchain==1.3.14" "langchain-groq==1.1.3" "langgraph==1.2.9" "python-dotenv==1.2.2" "ipykernel==7.3.0" "jupyter==1.1.1"

Note: you may need to restart the kernel to use updated packages.
